# Module 7: Data Cleaning

Run in Google Colab or a local PySpark environment.

In [ ]:
# Run this cell first
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("LearningModules").getOrCreate()
sc = spark.sparkContext

In [ ]:
# Load datasets (adjust path if needed)
students_df = spark.read.csv("../data/students.csv", header=True, inferSchema=True)
enrollments_df = spark.read.csv("../data/enrollments.csv", header=True, inferSchema=True)

## Loading Messy Data

Real-world data often has nulls, duplicates, inconsistent formatting, and invalid values.

In [ ]:
# Load the messy grades dataset
grades_messy = spark.read.csv("../data/grades_messy.csv", header=True, inferSchema=True)
grades_messy.show()
grades_messy.printSchema()

## Counting Nulls Per Column

The first step in cleaning is understanding where the problems are.

In [ ]:
# Count nulls in each column
null_counts = grades_messy.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in grades_messy.columns]
)
null_counts.show()

## Handling Nulls with dropna

In [ ]:
# Drop rows with any null values
print(f"Before dropna: {grades_messy.count()} rows")
cleaned = grades_messy.dropna()
print(f"After dropna (any): {cleaned.count()} rows")

# Drop rows only if specific column is null
cleaned_subset = grades_messy.dropna(subset=["grade"])
print(f"After dropna (grade only): {cleaned_subset.count()} rows")

## Removing Duplicates

In [ ]:
# Remove duplicate rows
print(f"Before dropDuplicates: {grades_messy.count()} rows")
deduped = grades_messy.dropDuplicates()
print(f"After dropDuplicates: {deduped.count()} rows")

# Remove duplicates based on specific columns
deduped_subset = grades_messy.dropDuplicates(["student_id", "course_id"])
print(f"After dropDuplicates (student_id, course_id): {deduped_subset.count()} rows")

## String Cleaning: trim and initcap

In [ ]:
# Trim whitespace and normalize capitalization
cleaned_names = students_df.withColumn("name_clean", initcap(trim(col("name"))))
cleaned_names.select("name", "name_clean").show(5)

## Filtering Invalid Values

In [ ]:
# Define valid grades
valid_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# Filter to only valid grades
valid_df = grades_messy.filter(col("grade").isin(valid_grades))
print(f"Valid grades only: {valid_df.count()} rows")
valid_df.show()

## Full Cleaning Pipeline

Combine all steps into a single pipeline.

In [ ]:
# Full cleaning pipeline
valid_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

cleaned_df = (
    grades_messy
    .dropDuplicates()
    .dropna(subset=["student_id", "course_id"])
    .withColumn("grade", upper(trim(col("grade"))))
    .filter(col("grade").isin(valid_grades))
)

print(f"Original: {grades_messy.count()} rows")
print(f"Cleaned:  {cleaned_df.count()} rows")
cleaned_df.show()

## Practice Problem 1: Null Handling

Fill null grades with 'N/A' instead of dropping them. Show the result.

<details><summary>Hint</summary>Use .fillna({"grade": "N/A"}) or .na.fill({"grade": "N/A"}).</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
filled_df = grades_messy.fillna({"grade": "N/A"})
filled_df.show()

## Practice Problem 2: String Normalization

On the grades_messy DataFrame, trim whitespace from the grade column, convert to uppercase, and show distinct values.

<details><summary>Hint</summary>Use withColumn with upper(trim(col("grade"))), then .select("grade").distinct().</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
normalized = grades_messy.withColumn("grade", upper(trim(col("grade"))))
normalized.select("grade").distinct().show()

## Practice Problem 3: Conditional Replacement

Replace any grade that is not in the valid_grades list with 'INVALID' rather than dropping the row.

<details><summary>Hint</summary>Use withColumn with when(col("grade").isin(valid_grades), col("grade")).otherwise("INVALID").</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
valid_grades = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

replaced_df = grades_messy.withColumn(
    "grade",
    when(upper(trim(col("grade"))).isin(valid_grades), upper(trim(col("grade"))))
    .otherwise("INVALID")
)
replaced_df.show()

## Practice Problem 4: Complete Pipeline

Write a cleaning pipeline that: removes duplicates, fills null student_ids with -1, trims and uppercases grades, and filters out rows where grade is null.

<details><summary>Hint</summary>Chain dropDuplicates, fillna, withColumn for trim/upper, and dropna(subset=["grade"]).</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
pipeline_df = (
    grades_messy
    .dropDuplicates()
    .fillna({"student_id": -1})
    .withColumn("grade", upper(trim(col("grade"))))
    .dropna(subset=["grade"])
)

print(f"Original: {grades_messy.count()} rows")
print(f"Cleaned:  {pipeline_df.count()} rows")
pipeline_df.show()